# Training-only electrode selection before CSP

This notebook tests whether selecting a smaller set of informative EEG electrodes improves the CSP neural network.

Three selection strategies are compared:

1. **Stimulus response:** electrodes with the most consistent mu/beta power change from the pre-cue baseline to the motor-imagery period.
2. **Left/right discrimination:** electrodes whose baseline-corrected power differs most consistently between left- and right-hand trials.
3. **Motor-region prior:** a fixed nine-electrode set around the sensorimotor cortex.

The data-driven strategies test the top 6, 9, 12, and 18 electrodes. The existing all-27-electrode pipeline is the reference.

Every data-dependent step—electrode ranking, CSP, scaling, and MLP fitting—is repeated inside participant-level cross-validation. Test participants are never loaded or evaluated.

## 1. Imports and experiment configuration

The pre-cue baseline is −2.5 to −0.5 seconds relative to the cue. The stimulus interval is 0.5 to 5.0 seconds, matching the CSP notebook while avoiding the immediate cue onset.

Mu is defined as 8–13 Hz and beta as 13–30 Hz. The MLP architecture remains `12 → 16 → 8 → 2`, so this experiment changes the electrodes—not the downstream model capacity.

In [1]:
from pathlib import Path
import copy
import random
import time

import mne
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy import linalg, signal
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score,
    confusion_matrix, f1_score, roc_auc_score,
)
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

mne.set_log_level('ERROR')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cpu')

def find_project_data(start=Path.cwd()):
    for parent in (start.resolve(), *start.resolve().parents):
        candidate = parent / 'data' / 'processed' / 'Signals'
        if candidate.is_dir():
            return parent / 'data'
    raise FileNotFoundError('Could not locate data/processed/Signals.')

DATA_ROOT = find_project_data()
SIGNALS_ROOT = DATA_ROOT / 'processed' / 'Signals'
INDEX_PATH = DATA_ROOT / 'processed' / 'modeling_index' / 'trial_modeling_index_with_split.csv'
RETAINED_PATH = (
    DATA_ROOT / 'processed' / 'all_trials_time_frequency_artifact_rejected'
    / 'retained_trials.csv'
)
SOURCE_COVARIANCE_ROOT = (
    DATA_ROOT / 'processed' / 'csp_neural_net' / 'covariances_by_file'
)
OUTPUT_ROOT = DATA_ROOT / 'processed' / 'csp_electrode_selection'
POWER_CACHE_ROOT = OUTPUT_ROOT / 'powers_by_file'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
POWER_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

ELECTRODES = [
    'Fz', 'FCz', 'Cz', 'CPz', 'Pz',
    'C1', 'C3', 'C5', 'C2', 'C4', 'C6',
    'F4', 'FC2', 'FC4', 'FC6', 'CP2', 'CP4', 'CP6', 'P4',
    'F3', 'FC1', 'FC3', 'FC5', 'CP1', 'CP3', 'CP5', 'P3',
]
BANDS_HZ = {'mu': (8.0, 13.0), 'beta': (13.0, 30.0)}
BASELINE_SECONDS = (-2.5, -0.5)
STIMULUS_SECONDS = (0.5, 5.0)
CHANNEL_COUNTS = [6, 9, 12, 18]
MOTOR_PRIOR = ['C3', 'C4', 'Cz', 'C1', 'C2', 'FC3', 'FC4', 'CP3', 'CP4']
N_CSP_FILTERS_PER_CLASS = 3
SHRINKAGE = 0.10
CV_FOLDS = 5
CV_EPOCHS = 7
BATCH_SIZE = 128

print('PyTorch:', torch.__version__)
print('Output:', OUTPUT_ROOT)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/aninanni/Documents/cosmos/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/aninanni/Documents/cosmos/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/Users/aninanni/Documents/cosmos/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 807, in st

PyTorch: 2.2.2
Output: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/csp_electrode_selection


## 2. Load development trials and verify participant isolation

Only the 55 training and 12 validation participants are loaded. The 12 test participants are excluded before any EEG file paths are processed.

In [2]:
model_index = pd.read_csv(INDEX_PATH)
development = model_index.loc[
    model_index['split'].isin(['train', 'validation'])
].copy()
retained = pd.read_csv(
    RETAINED_PATH, usecols=['source_file', 'trial', 'cue_sample']
)
development = development.merge(
    retained, on=['source_file', 'trial'], how='left', validate='one_to_one'
).reset_index(drop=True)

assert development['cue_sample'].notna().all()
assert set(development['split']) == {'train', 'validation'}
assert 'test' not in set(development['split'])
assert development['sample_id'].is_unique

participant_sets = {
    split: set(rows['participant'])
    for split, rows in development.groupby('split')
}
assert len(participant_sets['train']) == 55
assert len(participant_sets['validation']) == 12
assert participant_sets['train'].isdisjoint(participant_sets['validation'])

display(
    development.groupby('split').agg(
        participants=('participant', 'nunique'),
        recordings=('source_file', 'nunique'),
        trials=('sample_id', 'size'),
    ).reset_index()
)
print('Test trials loaded: 0')

,split,participants,recordings,trials
0,train,55,320,12156
1,validation,12,69,2570


Test trials loaded: 0


## 3. Load stimulus covariances and extract baseline/stimulus electrode power

The artifact-retained stimulus covariance matrices from `csp_neural_net.ipynb` are reused for CSP.

For electrode selection, each raw recording is filtered continuously into mu and beta. Mean squared amplitude is then calculated separately for the pre-cue and stimulus windows. Recording-level caches make the extraction resumable.

In [3]:
def source_covariance_path(source_file):
    relative = Path(source_file).with_suffix('')
    return (
        SOURCE_COVARIANCE_ROOT / relative.parent
        / f'{relative.name}_covariances.npz'
    )

def power_cache_path(source_file):
    relative = Path(source_file).with_suffix('')
    return (
        POWER_CACHE_ROOT / relative.parent
        / f'{relative.name}_baseline_stimulus_power.npz'
    )

def power_cache_valid(path, expected_ids):
    if not path.is_file():
        return False
    try:
        with np.load(path, allow_pickle=False) as saved:
            return (
                saved['powers'].shape == (len(expected_ids), 2, 2, 27)
                and np.array_equal(saved['sample_ids'].astype(str), expected_ids)
                and np.isfinite(saved['powers']).all()
                and np.all(saved['powers'] > 0)
            )
    except Exception:
        return False

def extract_power_file(source_file, rows):
    rows = rows.sort_values('sample_id').reset_index(drop=True)
    sample_ids = rows['sample_id'].to_numpy(dtype=str)
    destination = power_cache_path(source_file)
    if power_cache_valid(destination, sample_ids):
        return destination, 'cached'

    raw = mne.io.read_raw_gdf(
        SIGNALS_ROOT / source_file, preload=True, verbose='ERROR'
    )
    try:
        sfreq = float(raw.info['sfreq'])
        assert sfreq == 512.0
        eeg = raw.get_data(picks=ELECTRODES)
    finally:
        raw.close()

    filtered_bands = []
    for low_hz, high_hz in BANDS_HZ.values():
        sos = signal.butter(
            4, [low_hz, high_hz], btype='bandpass', fs=sfreq, output='sos'
        )
        filtered_bands.append(signal.sosfiltfilt(sos, eeg, axis=1))
    del eeg

    powers = np.empty((len(rows), 2, 2, 27), dtype=np.float32)
    for row_number, row in enumerate(rows.itertuples(index=False)):
        cue = int(row.cue_sample)
        windows = [
            (
                cue + int(round(BASELINE_SECONDS[0] * sfreq)),
                cue + int(round(BASELINE_SECONDS[1] * sfreq)),
            ),
            (
                cue + int(round(STIMULUS_SECONDS[0] * sfreq)),
                cue + int(round(STIMULUS_SECONDS[1] * sfreq)),
            ),
        ]
        for band_number, filtered in enumerate(filtered_bands):
            for window_number, (start, stop) in enumerate(windows):
                if start < 0 or stop > filtered.shape[1] or stop <= start:
                    raise ValueError(f'Invalid epoch bounds for {row.sample_id}.')
                epoch = filtered[:, start:stop]
                epoch = epoch - epoch.mean(axis=1, keepdims=True)
                powers[row_number, band_number, window_number] = np.mean(
                    np.square(epoch), axis=1
                )

    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + '.tmp.npz')
    np.savez_compressed(
        temporary,
        powers=powers,
        sample_ids=sample_ids,
        bands=np.asarray(list(BANDS_HZ)),
        windows=np.asarray(['baseline', 'stimulus']),
        electrodes=np.asarray(ELECTRODES),
    )
    temporary.replace(destination)
    return destination, 'computed'

all_covariances = np.empty((len(development), 2, 27, 27), dtype=np.float32)
all_powers = np.empty((len(development), 2, 2, 27), dtype=np.float32)
position_by_sample = pd.Series(
    np.arange(len(development)), index=development['sample_id']
)
file_groups = list(development.groupby('source_file', sort=True))
status_counts = {'computed': 0, 'cached': 0}
started = time.perf_counter()

for file_number, (source_file, rows) in enumerate(file_groups, start=1):
    ordered_ids = rows.sort_values('sample_id')['sample_id'].to_numpy(dtype=str)
    covariance_path = source_covariance_path(source_file)
    if not covariance_path.is_file():
        raise FileNotFoundError(
            f'Missing CSP covariance cache: {covariance_path}. '
            'Run csp_neural_net.ipynb first.'
        )
    with np.load(covariance_path, allow_pickle=False) as saved:
        covariance_ids = saved['sample_ids'].astype(str)
        covariances = saved['covariances']
    assert np.array_equal(covariance_ids, ordered_ids)

    power_path, status = extract_power_file(source_file, rows)
    status_counts[status] += 1
    with np.load(power_path, allow_pickle=False) as saved:
        power_ids = saved['sample_ids'].astype(str)
        powers = saved['powers']
    assert np.array_equal(power_ids, ordered_ids)

    positions = position_by_sample.loc[ordered_ids].to_numpy(dtype=int)
    all_covariances[positions] = covariances
    all_powers[positions] = powers
    if file_number == 1 or file_number % 25 == 0 or file_number == len(file_groups):
        print(
            f'[{file_number:>3}/{len(file_groups)}] {status:<8} '
            f'({(time.perf_counter() - started) / 60:.1f} min)'
        )

assert np.isfinite(all_covariances).all()
assert np.isfinite(all_powers).all() and np.all(all_powers > 0)
print('Power-cache status:', status_counts)
print('Test recordings opened: 0')

[  1/389] cached   (0.0 min)
[ 25/389] cached   (0.0 min)


[ 50/389] cached   (0.0 min)
[ 75/389] cached   (0.0 min)


[100/389] cached   (0.0 min)
[125/389] cached   (0.0 min)


[150/389] cached   (0.0 min)
[175/389] cached   (0.0 min)


[200/389] cached   (0.0 min)
[225/389] cached   (0.0 min)


[250/389] cached   (0.0 min)
[275/389] cached   (0.0 min)


[300/389] cached   (0.0 min)


[325/389] cached   (0.0 min)
[350/389] cached   (0.0 min)
[375/389] cached   (0.0 min)


[389/389] cached   (0.0 min)
Power-cache status: {'computed': 0, 'cached': 389}
Test recordings opened: 0


## 4. Define leakage-safe electrode ranking, CSP, and MLP helpers

Stimulus-response ranking uses participant-level median log power ratios and favors large, consistent changes across participants.

Left/right ranking uses the participant-level difference between median left and right baseline-corrected power. It is supervised, so it must be fitted inside each training fold.

In [4]:
y_all = development['label_id'].to_numpy(dtype=np.int64)
train_mask = development['split'].eq('train').to_numpy()
validation_mask = development['split'].eq('validation').to_numpy()
participant_array = development['participant'].to_numpy()
log_power_ratio = np.log(
    all_powers[:, :, 1, :] / all_powers[:, :, 0, :]
)

def stable_effect_score(participant_effects):
    mean_effect = participant_effects.mean(axis=0)
    standard_deviation = participant_effects.std(axis=0, ddof=1)
    return np.abs(mean_effect) / np.maximum(standard_deviation, 1e-8)

def rank_task_response(fit_mask):
    effects = []
    for participant in sorted(set(participant_array[fit_mask])):
        rows = fit_mask & (participant_array == participant)
        effects.append(np.median(log_power_ratio[rows], axis=0))
    band_scores = stable_effect_score(np.stack(effects))
    electrode_scores = band_scores.mean(axis=0)
    return np.argsort(-electrode_scores), electrode_scores

def rank_left_right(fit_mask):
    effects = []
    for participant in sorted(set(participant_array[fit_mask])):
        participant_rows = fit_mask & (participant_array == participant)
        left = np.median(log_power_ratio[participant_rows & (y_all == 0)], axis=0)
        right = np.median(log_power_ratio[participant_rows & (y_all == 1)], axis=0)
        effects.append(left - right)
    band_scores = stable_effect_score(np.stack(effects))
    electrode_scores = band_scores.mean(axis=0)
    return np.argsort(-electrode_scores), electrode_scores

def shrink_covariance(covariance):
    target = np.eye(len(covariance)) * np.trace(covariance) / len(covariance)
    return (1.0 - SHRINKAGE) * covariance + SHRINKAGE * target

def fit_csp(covariances, labels):
    left = shrink_covariance(covariances[labels == 0].mean(axis=0))
    right = shrink_covariance(covariances[labels == 1].mean(axis=0))
    eigenvalues, eigenvectors = linalg.eigh(left, left + right)
    selected = np.r_[
        np.arange(N_CSP_FILTERS_PER_CLASS),
        np.arange(
            len(eigenvalues) - N_CSP_FILTERS_PER_CLASS,
            len(eigenvalues),
        ),
    ]
    return eigenvectors[:, selected].T

def csp_features(covariances, filters):
    variances = np.einsum(
        'kc,ncd,kd->nk', filters, covariances, filters, optimize=True
    )
    variances = np.maximum(variances, np.finfo(float).eps)
    return np.log(variances / variances.sum(axis=1, keepdims=True))

def build_features(fit_mask, electrode_indices):
    blocks = []
    selected_covariances = all_covariances[
        :, :, electrode_indices, :
    ][:, :, :, electrode_indices]
    for band_number in range(2):
        filters = fit_csp(
            selected_covariances[fit_mask, band_number], y_all[fit_mask]
        )
        blocks.append(
            csp_features(selected_covariances[:, band_number], filters)
        )
    raw_features = np.column_stack(blocks).astype(np.float32)
    mean = raw_features[fit_mask].mean(axis=0, dtype=np.float64)
    std = raw_features[fit_mask].std(axis=0, dtype=np.float64)
    assert np.all(std > 0)
    return ((raw_features - mean) / std).astype(np.float32)

class ElectrodeSelectedCSPMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(12, 16), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(16, 8), nn.ReLU(), nn.Dropout(0.10),
            nn.Linear(8, 2),
        )

    def forward(self, inputs):
        return self.network(inputs)

def to_tensor(values, dtype):
    values = np.ascontiguousarray(values)
    return torch.frombuffer(memoryview(values), dtype=dtype).reshape(values.shape)

def fit_fixed_epoch_mlp(X, fit_mask, epochs, seed):
    X_tensor = to_tensor(X[fit_mask], torch.float32)
    y_tensor = to_tensor(y_all[fit_mask], torch.int64)
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        TensorDataset(X_tensor, y_tensor), batch_size=BATCH_SIZE,
        shuffle=True, generator=generator,
    )
    torch.manual_seed(seed)
    model = ElectrodeSelectedCSPMLP().to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=1e-3, weight_decay=1e-4
    )
    loss_function = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for batch_X, batch_y in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_function(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()
    return model

@torch.no_grad()
def predict_model(model, X, mask):
    X_tensor = to_tensor(X[mask], torch.float32)
    y_tensor = to_tensor(y_all[mask], torch.int64)
    loader = DataLoader(
        TensorDataset(X_tensor, y_tensor), batch_size=BATCH_SIZE, shuffle=False
    )
    model.eval()
    predictions, probabilities = [], []
    for batch_X, _ in loader:
        logits = model(batch_X)
        predictions.extend(logits.argmax(dim=1).tolist())
        probabilities.extend(torch.softmax(logits, dim=1)[:, 1].tolist())
    return np.asarray(predictions), np.asarray(probabilities)

print('Selection and modeling helpers ready.')

Selection and modeling helpers ready.


## 5. Compare electrode strategies using held-out training participants

The 55 training participants are divided into five non-overlapping folds. For each fold and candidate:

1. Rank electrodes using the other four folds only.
2. Fit CSP and scaling using those same participants only.
3. Train the MLP for seven fixed epochs.
4. Predict the held-out participants.

This produces an out-of-fold prediction for every training trial without using the validation or test set to choose the electrode strategy.

In [5]:
training_participants = np.asarray(sorted(participant_sets['train']))
rng = np.random.default_rng(SEED)
fold_assignment = {
    participant: fold
    for fold, members in enumerate(
        np.array_split(rng.permutation(training_participants), CV_FOLDS), start=1
    )
    for participant in members
}
motor_prior_indices = np.asarray([ELECTRODES.index(name) for name in MOTOR_PRIOR])

candidates = [
    {'candidate': f'task_response_top_{count}', 'method': 'task_response', 'count': count}
    for count in CHANNEL_COUNTS
] + [
    {'candidate': f'left_right_top_{count}', 'method': 'left_right', 'count': count}
    for count in CHANNEL_COUNTS
] + [
    {'candidate': 'motor_prior_9', 'method': 'motor_prior', 'count': 9},
    {'candidate': 'all_27', 'method': 'all', 'count': 27},
]

prediction_frames = []
fold_selection_rows = []
comparison_started = time.perf_counter()

for fold in range(1, CV_FOLDS + 1):
    held_out_participants = {
        participant for participant, assigned in fold_assignment.items()
        if assigned == fold
    }
    holdout_mask = train_mask & np.isin(participant_array, list(held_out_participants))
    fit_mask = train_mask & ~holdout_mask
    assert set(participant_array[fit_mask]).isdisjoint(held_out_participants)

    task_order, task_scores = rank_task_response(fit_mask)
    supervised_order, supervised_scores = rank_left_right(fit_mask)

    for candidate_number, candidate in enumerate(candidates, start=1):
        if candidate['method'] == 'task_response':
            indices = task_order[:candidate['count']]
            scores = task_scores
        elif candidate['method'] == 'left_right':
            indices = supervised_order[:candidate['count']]
            scores = supervised_scores
        elif candidate['method'] == 'motor_prior':
            indices = motor_prior_indices
            scores = np.full(27, np.nan)
        else:
            indices = np.arange(27)
            scores = np.full(27, np.nan)

        X_candidate = build_features(fit_mask, indices)
        fold_model = fit_fixed_epoch_mlp(
            X_candidate, fit_mask, CV_EPOCHS,
            SEED + fold * 100,
        )
        predictions, probabilities = predict_model(
            fold_model, X_candidate, holdout_mask
        )
        rows = development.loc[holdout_mask, [
            'sample_id', 'participant', 'label_id'
        ]].reset_index(drop=True)
        rows['prediction'] = predictions
        rows['right_probability'] = probabilities
        rows['fold'] = fold
        rows['candidate'] = candidate['candidate']
        prediction_frames.append(rows)
        fold_selection_rows.append({
            'fold': fold,
            'candidate': candidate['candidate'],
            'method': candidate['method'],
            'electrode_count': len(indices),
            'selected_electrodes': '|'.join(np.asarray(ELECTRODES)[indices]),
        })

    print(
        f'Fold {fold}/{CV_FOLDS} complete: '
        f'{len(held_out_participants)} held-out participants '
        f'({(time.perf_counter() - comparison_started) / 60:.1f} min)'
    )

oof_predictions = pd.concat(prediction_frames, ignore_index=True)
fold_selections = pd.DataFrame(fold_selection_rows)
assert oof_predictions.groupby('candidate')['sample_id'].nunique().eq(
    int(train_mask.sum())
).all()

summary_rows = []
for candidate, rows in oof_predictions.groupby('candidate'):
    summary_rows.append({
        'candidate': candidate,
        'method': next(item['method'] for item in candidates if item['candidate'] == candidate),
        'electrode_count': next(item['count'] for item in candidates if item['candidate'] == candidate),
        'trials': len(rows),
        'accuracy': accuracy_score(rows['label_id'], rows['prediction']),
        'balanced_accuracy': balanced_accuracy_score(
            rows['label_id'], rows['prediction']
        ),
        'macro_f1': f1_score(rows['label_id'], rows['prediction'], average='macro'),
        'roc_auc': roc_auc_score(rows['label_id'], rows['right_probability']),
    })

cv_summary = pd.DataFrame(summary_rows).sort_values(
    ['balanced_accuracy', 'electrode_count'], ascending=[False, True]
).reset_index(drop=True)
display(cv_summary)
display(fold_selections)

oof_predictions.to_csv(OUTPUT_ROOT / 'cross_validation_predictions.csv', index=False)
fold_selections.to_csv(OUTPUT_ROOT / 'fold_electrode_selections.csv', index=False)
cv_summary.to_csv(OUTPUT_ROOT / 'strategy_comparison.csv', index=False)

best_candidate = cv_summary.iloc[0].to_dict()
print('Cross-validation winner:', best_candidate['candidate'])

Fold 1/5 complete: 11 held-out participants (0.1 min)


Fold 2/5 complete: 11 held-out participants (0.1 min)


Fold 3/5 complete: 11 held-out participants (0.2 min)


Fold 4/5 complete: 11 held-out participants (0.3 min)


Fold 5/5 complete: 11 held-out participants (0.3 min)


,candidate,method,electrode_count,trials,accuracy,balanced_accuracy,macro_f1,roc_auc
0,all_27,all,27,12156,0.661649,0.661566,0.661433,0.729311
1,left_right_top_18,left_right,18,12156,0.651201,0.651192,0.651193,0.718032
2,task_response_top_18,task_response,18,12156,0.647252,0.647183,0.647088,0.707053
3,motor_prior_9,motor_prior,9,12156,0.643551,0.643581,0.643537,0.698618
4,left_right_top_12,left_right,12,12156,0.637463,0.637352,0.637068,0.703780
5,task_response_top_12,task_response,12,12156,0.634831,0.634778,0.634727,0.695450
6,task_response_top_9,task_response,9,12156,0.621915,0.621748,0.621022,0.676368
7,task_response_top_6,task_response,6,12156,0.611550,0.611484,0.611388,0.658358
8,left_right_top_9,left_right,9,12156,0.605545,0.605191,0.601564,0.663079
9,left_right_top_6,left_right,6,12156,0.599622,0.599326,0.596788,0.656820


,fold,candidate,method,electrode_count,selected_electrodes
0,1,task_response_top_6,task_response,6,Fz|C2|FC2|C3|FCz|Cz
1,1,task_response_top_9,task_response,9,Fz|C2|FC2|C3|FCz|Cz|CP1|FC4|C1
2,1,task_response_top_12,task_response,12,Fz|C2|FC2|C3|FCz|Cz|CP1|FC4|C1|CP2|F4|CPz
3,1,task_response_top_18,task_response,18,Fz|C2|FC2|C3|FCz|Cz|CP1|FC4|C1|CP2|F4|CPz|FC1|...
4,1,left_right_top_6,left_right,6,C3|C4|CP3|C5|FC4|C1
5,1,left_right_top_9,left_right,9,C3|C4|CP3|C5|FC4|C1|CP5|C6|CP6
6,1,left_right_top_12,left_right,12,C3|C4|CP3|C5|FC4|C1|CP5|C6|CP6|CP4|CP1|FC3
7,1,left_right_top_18,left_right,18,C3|C4|CP3|C5|FC4|C1|CP5|C6|CP6|CP4|CP1|FC3|FC6...
8,1,motor_prior_9,motor_prior,9,C3|C4|Cz|C1|C2|FC3|FC4|CP3|CP4
9,1,all_27,all,27,Fz|FCz|Cz|CPz|Pz|C1|C3|C5|C2|C4|C6|F4|FC2|FC4|...


Cross-validation winner: all_27


## 6. Fit the winning electrode selection on all training participants

The winning method and channel count are now frozen. Its electrode ranking is recalculated using all 55 training participants, then CSP and scaling are fitted on those participants.

The validation participants are used only for early stopping and final development-stage reporting. The test split remains untouched.

In [6]:
winning_method = best_candidate['method']
winning_count = int(best_candidate['electrode_count'])

# Show both complete training-only rankings even if all 27 electrodes win.
task_final_order, task_final_scores = rank_task_response(train_mask)
supervised_final_order, supervised_final_scores = rank_left_right(train_mask)
task_ranks = np.empty(27, dtype=int)
supervised_ranks = np.empty(27, dtype=int)
task_ranks[task_final_order] = np.arange(1, 28)
supervised_ranks[supervised_final_order] = np.arange(1, 28)
ranking_comparison = pd.DataFrame({
    'electrode': ELECTRODES,
    'task_response_rank': task_ranks,
    'task_response_score': task_final_scores,
    'left_right_rank': supervised_ranks,
    'left_right_score': supervised_final_scores,
}).sort_values('task_response_rank')
display(ranking_comparison)

if winning_method == 'task_response':
    final_order, final_scores = task_final_order, task_final_scores
    selected_indices = final_order[:winning_count]
elif winning_method == 'left_right':
    final_order, final_scores = supervised_final_order, supervised_final_scores
    selected_indices = final_order[:winning_count]
elif winning_method == 'motor_prior':
    selected_indices = motor_prior_indices
    final_scores = np.full(27, np.nan)
else:
    selected_indices = np.arange(27)
    final_scores = np.full(27, np.nan)

selected_electrodes = np.asarray(ELECTRODES)[selected_indices].tolist()
ranking_table = pd.DataFrame({
    'electrode': ELECTRODES,
    'selection_score': final_scores,
    'selected': np.isin(np.arange(27), selected_indices),
}).sort_values(['selected', 'selection_score'], ascending=[False, False])
display(ranking_table)
print('Selected electrodes:', selected_electrodes)

X_final = build_features(train_mask, selected_indices)
X_train = to_tensor(X_final[train_mask], torch.float32)
y_train = to_tensor(y_all[train_mask], torch.int64)
X_validation = to_tensor(X_final[validation_mask], torch.float32)
y_validation = to_tensor(y_all[validation_mask], torch.int64)

generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    TensorDataset(X_train, y_train), batch_size=BATCH_SIZE,
    shuffle=True, generator=generator,
)
validation_loader = DataLoader(
    TensorDataset(X_validation, y_validation),
    batch_size=BATCH_SIZE, shuffle=False,
)

torch.manual_seed(SEED)
final_model = ElectrodeSelectedCSPMLP().to(DEVICE)
optimizer = torch.optim.AdamW(
    final_model.parameters(), lr=1e-3, weight_decay=1e-4
)
loss_function = nn.CrossEntropyLoss()
MAX_EPOCHS = 100
PATIENCE = 12
MIN_IMPROVEMENT = 1e-4

@torch.no_grad()
def evaluate_loader(model, loader):
    model.eval()
    targets, predictions, probabilities = [], [], []
    loss_total = 0.0
    for batch_X, batch_y in loader:
        logits = model(batch_X)
        loss_total += loss_function(logits, batch_y).item() * len(batch_X)
        targets.extend(batch_y.tolist())
        predictions.extend(logits.argmax(dim=1).tolist())
        probabilities.extend(torch.softmax(logits, dim=1)[:, 1].tolist())
    return (
        np.asarray(targets), np.asarray(predictions), np.asarray(probabilities),
        loss_total / len(loader.dataset),
    )

history_rows = []
best_state = None
best_epoch = 0
best_validation_loss = np.inf
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    final_model.train()
    train_loss_total = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad(set_to_none=True)
        logits = final_model(batch_X)
        loss = loss_function(logits, batch_y)
        loss.backward()
        optimizer.step()
        train_loss_total += loss.item() * len(batch_X)
    train_loss = train_loss_total / len(train_loader.dataset)
    validation_targets, validation_predictions, validation_probabilities, validation_loss = (
        evaluate_loader(final_model, validation_loader)
    )
    history_rows.append({
        'epoch': epoch,
        'training_loss': train_loss,
        'validation_loss': validation_loss,
        'validation_balanced_accuracy': balanced_accuracy_score(
            validation_targets, validation_predictions
        ),
        'validation_roc_auc': roc_auc_score(
            validation_targets, validation_probabilities
        ),
    })
    if validation_loss < best_validation_loss - MIN_IMPROVEMENT:
        best_validation_loss = validation_loss
        best_epoch = epoch
        best_state = copy.deepcopy(final_model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

final_model.load_state_dict(best_state)
train_result = evaluate_loader(final_model, train_loader)
validation_result = evaluate_loader(final_model, validation_loader)

def metric_row(split, result):
    targets, predictions, probabilities, loss = result
    matrix = confusion_matrix(targets, predictions, labels=[0, 1])
    return {
        'split': split,
        'participants': len(participant_sets[split]),
        'trials': len(targets),
        'loss': loss,
        'accuracy': accuracy_score(targets, predictions),
        'balanced_accuracy': balanced_accuracy_score(targets, predictions),
        'macro_f1': f1_score(targets, predictions, average='macro'),
        'roc_auc': roc_auc_score(targets, probabilities),
        'cohen_kappa': cohen_kappa_score(targets, predictions),
        'true_left_pred_left': int(matrix[0, 0]),
        'true_left_pred_right': int(matrix[0, 1]),
        'true_right_pred_left': int(matrix[1, 0]),
        'true_right_pred_right': int(matrix[1, 1]),
    }

final_metrics = pd.DataFrame([
    metric_row('train', train_result),
    metric_row('validation', validation_result),
])
display(final_metrics)

comparison_with_current = cv_summary.copy()
comparison_with_current['selected_by_cv'] = (
    comparison_with_current['candidate'] == best_candidate['candidate']
)
display(comparison_with_current)

pd.DataFrame(history_rows).to_csv(
    OUTPUT_ROOT / 'final_training_history.csv', index=False
)
final_metrics.to_csv(OUTPUT_ROOT / 'final_training_validation_metrics.csv', index=False)
ranking_table.to_csv(OUTPUT_ROOT / 'final_electrode_ranking.csv', index=False)
ranking_comparison.to_csv(
    OUTPUT_ROOT / 'training_only_electrode_rankings.csv', index=False
)

torch.save({
    'model_state_dict': final_model.state_dict(),
    'architecture': '12 -> 16 -> 8 -> 2',
    'selection_method': winning_method,
    'selected_electrodes': selected_electrodes,
    'best_epoch': best_epoch,
    'trained': True,
    'tested': False,
    'seed': SEED,
}, OUTPUT_ROOT / 'trained_electrode_selected_csp_mlp.pt')

print('Winning candidate:', best_candidate['candidate'])
print('Selected electrodes:', selected_electrodes)
print('Best final epoch:', best_epoch)
print('Test split evaluated: NO')

,electrode,task_response_rank,task_response_score,left_right_rank,left_right_score
6,C3,1,0.752015,1,0.479307
8,C2,2,0.751979,17,0.174504
23,CP1,3,0.735391,9,0.353960
5,C1,4,0.733845,8,0.367185
13,FC4,5,0.732386,3,0.466931
2,Cz,6,0.729643,24,0.111807
15,CP2,7,0.720363,27,0.042268
12,FC2,8,0.714719,22,0.154029
21,FC3,9,0.705071,13,0.235213
0,Fz,10,0.704295,18,0.169440


,electrode,selection_score,selected
0,Fz,NaN,True
1,FCz,NaN,True
2,Cz,NaN,True
3,CPz,NaN,True
4,Pz,NaN,True
5,C1,NaN,True
6,C3,NaN,True
7,C5,NaN,True
8,C2,NaN,True
9,C4,NaN,True


Selected electrodes: ['Fz', 'FCz', 'Cz', 'CPz', 'Pz', 'C1', 'C3', 'C5', 'C2', 'C4', 'C6', 'F4', 'FC2', 'FC4', 'FC6', 'CP2', 'CP4', 'CP6', 'P4', 'F3', 'FC1', 'FC3', 'FC5', 'CP1', 'CP3', 'CP5', 'P3']


Early stopping at epoch 19.


,split,participants,trials,loss,accuracy,balanced_accuracy,macro_f1,roc_auc,cohen_kappa,true_left_pred_left,true_left_pred_right,true_right_pred_left,true_right_pred_right
0,train,55,12156,0.580840,0.672507,0.672529,0.672503,0.752589,0.345040,4066,2034,1947,4109
1,validation,12,2570,0.620646,0.652529,0.652380,0.651948,0.706625,0.304848,891,399,494,786


,candidate,method,electrode_count,trials,accuracy,balanced_accuracy,macro_f1,roc_auc,selected_by_cv
0,all_27,all,27,12156,0.661649,0.661566,0.661433,0.729311,True
1,left_right_top_18,left_right,18,12156,0.651201,0.651192,0.651193,0.718032,False
2,task_response_top_18,task_response,18,12156,0.647252,0.647183,0.647088,0.707053,False
3,motor_prior_9,motor_prior,9,12156,0.643551,0.643581,0.643537,0.698618,False
4,left_right_top_12,left_right,12,12156,0.637463,0.637352,0.637068,0.703780,False
5,task_response_top_12,task_response,12,12156,0.634831,0.634778,0.634727,0.695450,False
6,task_response_top_9,task_response,9,12156,0.621915,0.621748,0.621022,0.676368,False
7,task_response_top_6,task_response,6,12156,0.611550,0.611484,0.611388,0.658358,False
8,left_right_top_9,left_right,9,12156,0.605545,0.605191,0.601564,0.663079,False
9,left_right_top_6,left_right,6,12156,0.599622,0.599326,0.596788,0.656820,False


Winning candidate: all_27
Selected electrodes: ['Fz', 'FCz', 'Cz', 'CPz', 'Pz', 'C1', 'C3', 'C5', 'C2', 'C4', 'C6', 'F4', 'FC2', 'FC4', 'FC6', 'CP2', 'CP4', 'CP6', 'P4', 'F3', 'FC1', 'FC3', 'FC5', 'CP1', 'CP3', 'CP5', 'P3']
Best final epoch: 7
Test split evaluated: NO


## Interpretation guardrails

- A smaller electrode set is useful only if its participant-level cross-validation performance is at least as good as the all-27 reference.
- Stimulus variance alone is not assumed to be discriminative; that is why the task-response ranking is compared with supervised ranking and a motor-region prior.
- The validation score remains a development result because validation loss selected the stopping epoch.
- No conclusions about final generalization should be made until the chosen pipeline is frozen and evaluated once on the untouched test participants.